# AdiVaani NMT — Part I: Random Embeddings (LSTM Seq2Seq + Attention)
**Run this FIRST. It uses T4x2 and DataParallel. Target: ~3-4 hours.**

Architecture:
- Bidirectional LSTM Encoder
- LSTM Decoder with Bahdanau Attention
- BPE Tokenization via SentencePiece (shared vocab)
- Mixed precision (AMP), DataParallel, packed sequences
- Label smoothing, beam search (width=4)
- Outputs: loss curves, BLEU-100, CHRF++-100

In [ ]:
pip install sacrebleu -q

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies safely
# ─────────────────────────────────────────────────────────────────────────────
import importlib
import subprocess
import sys

pkgs = ['sentencepiece', 'sacrebleu', 'matplotlib', 'tqdm']

for p in pkgs:
    try:
        importlib.import_module(p)
        print(f"{p} already installed")
    except ImportError:
        print(f"Installing {p}...")
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', p]
        )

print("Done.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import os, random, math, time, json, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.amp import autocast, GradScaler
import sentencepiece as spm
import sacrebleu
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device: {DEVICE} | GPUs: {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Configuration (tune here)
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Data
    'data_dir': '/kaggle/input/datasets/vidhijjain/adivani',          # update to your dataset path
    'hi_file': 'train.hi',               # Hindi file name
    'mr_file': 'train.mr',               # Marathi file name
    'max_samples': 300_000,              # subsample for speed; use None for full
    'max_len': 60,                       # max tokens per sentence
    'val_split': 0.05,

    # Tokenization
    'vocab_size': 8000,                  # shared BPE vocab
    'spm_model': '/kaggle/working/spm_shared',

    # Model
    'embed_dim': 256,
    'hidden_dim': 512,
    'n_layers': 2,
    'dropout': 0.3,
    'bidirectional': True,

    # Training
    'batch_size': 256,                   # per GPU; effective = 256 * N_GPUS
    'epochs': 20,
    'lr': 3e-3,
    'lr_patience': 2,
    'clip_grad': 1.0,
    'label_smooth': 0.1,
    'accum_steps': 1,                    # gradient accumulation
    'warmup_steps': 500,

    # Decoding
    'beam_width': 4,
    'len_penalty': 0.6,

    # Eval
    'eval_every': 1,                     # eval after every N epochs
    'bleu_samples': 1000,                # subsample for fast BLEU eval

    # Output
    'ckpt_path': '/kaggle/working/best_random.pt',
    'out_dir': '/kaggle/working',
}

# Auto-detect dataset files (common Kaggle layouts)
import glob
search = glob.glob('kaggle/input/datasets/vidhijjain/adivanits/*.hi', recursive=True)
if search:
    CFG['hi_file'] = os.path.basename(search[0])
    CFG['data_dir'] = os.path.dirname(search[0])
    CFG['mr_file'] = CFG['hi_file'].replace('.hi', '.mr')
    print(f'Auto-detected: {search[0]}')
else:
    # Try txt / tsv formats
    search_tsv = glob.glob('/kaggle/input/**/*.tsv', recursive=True)
    search_txt = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    print('Dataset files found:', search_tsv + search_txt)
    print('Please update CFG[data_dir], CFG[hi_file], CFG[mr_file] manually.')

print('Config ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Load & inspect data
# ─────────────────────────────────────────────────────────────────────────────
def load_parallel(hi_path, mr_path, max_samples=None, max_len=100):
    src_lines, tgt_lines = [], []
    with open(hi_path, encoding='utf-8') as f_hi, \
         open(mr_path, encoding='utf-8') as f_mr:
        for hi, mr in zip(f_hi, f_mr):
            hi, mr = hi.strip(), mr.strip()
            if hi and mr and len(hi.split()) <= max_len and len(mr.split()) <= max_len:
                src_lines.append(hi)
                tgt_lines.append(mr)
    if max_samples and len(src_lines) > max_samples:
        idx = random.sample(range(len(src_lines)), max_samples)
        src_lines = [src_lines[i] for i in idx]
        tgt_lines = [tgt_lines[i] for i in idx]
    return src_lines, tgt_lines

hi_path = os.path.join(CFG['data_dir'], CFG['hi_file'])
mr_path = os.path.join(CFG['data_dir'], CFG['mr_file'])

src_lines, tgt_lines = load_parallel(hi_path, mr_path,
                                      CFG['max_samples'], CFG['max_len'])
print(f'Loaded {len(src_lines):,} sentence pairs')
print(f'Sample SRC: {src_lines[0]}')
print(f'Sample TGT: {tgt_lines[0]}')

# Train / Val split
n_val = int(len(src_lines) * CFG['val_split'])
n_train = len(src_lines) - n_val
indices = list(range(len(src_lines)))
random.shuffle(indices)
train_idx, val_idx = indices[:n_train], indices[n_train:]

train_src = [src_lines[i] for i in train_idx]
train_tgt = [tgt_lines[i] for i in train_idx]
val_src   = [src_lines[i] for i in val_idx]
val_tgt   = [tgt_lines[i] for i in val_idx]
print(f'Train: {len(train_src):,} | Val: {len(val_src):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Train SentencePiece (shared BPE, both Hindi + Marathi)
# ─────────────────────────────────────────────────────────────────────────────
raw_corpus = '/kaggle/working/spm_input.txt'
with open(raw_corpus, 'w', encoding='utf-8') as f:
    for line in src_lines + tgt_lines:
        f.write(line + '\n')

spm.SentencePieceTrainer.train(
    input=raw_corpus,
    model_prefix=CFG['spm_model'],
    vocab_size=CFG['vocab_size'],
    character_coverage=0.9995,
    model_type='bpe',
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    shuffle_input_sentence=True,
    input_sentence_size=500_000,
)

sp = spm.SentencePieceProcessor()
sp.load(CFG['spm_model'] + '.model')

PAD_ID = sp.pad_id()   # 0
BOS_ID = sp.bos_id()   # 2
EOS_ID = sp.eos_id()   # 3
UNK_ID = sp.unk_id()   # 1
VOCAB_SIZE = sp.get_piece_size()

print(f'Vocab size: {VOCAB_SIZE}')
print('Sample tokenization:', sp.encode('मैं घर जा रहा हूँ।', out_type=str))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Dataset & DataLoader
# ─────────────────────────────────────────────────────────────────────────────
class NMTDataset(Dataset):
    def __init__(self, src_lines, tgt_lines, sp, max_len):
        self.pairs = []
        for s, t in zip(src_lines, tgt_lines):
            src_ids = sp.encode(s)
            tgt_ids = sp.encode(t)
            if 1 <= len(src_ids) <= max_len and 1 <= len(tgt_ids) <= max_len:
                self.pairs.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def collate_fn(batch):
    """Sort by src length (descending) for pack_padded_sequence."""
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    src_seqs, tgt_seqs = zip(*batch)

    src_lens = [len(s) for s in src_seqs]
    tgt_lens = [len(t) + 1 for t in tgt_seqs]  # +1 for EOS

    src_pad = torch.zeros(len(src_seqs), max(src_lens), dtype=torch.long)
    tgt_in  = torch.zeros(len(tgt_seqs), max(tgt_lens), dtype=torch.long)
    tgt_out = torch.full((len(tgt_seqs), max(tgt_lens)), PAD_ID, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src_seqs, tgt_seqs)):
        src_pad[i, :len(s)] = torch.tensor(s)
        tgt_with_bos = [BOS_ID] + t
        tgt_with_eos = t + [EOS_ID]
        tgt_in[i,  :len(tgt_with_bos)] = torch.tensor(tgt_with_bos)
        tgt_out[i, :len(tgt_with_eos)] = torch.tensor(tgt_with_eos)

    return src_pad, torch.tensor(src_lens), tgt_in, tgt_out


train_ds = NMTDataset(train_src, train_tgt, sp, CFG['max_len'])
val_ds   = NMTDataset(val_src,   val_tgt,   sp, CFG['max_len'])

effective_batch = CFG['batch_size'] * max(N_GPUS, 1)
train_loader = DataLoader(train_ds, batch_size=effective_batch, shuffle=True,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=effective_batch * 2, shuffle=False,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)

print(f'Train dataset: {len(train_ds):,} | Val dataset: {len(val_ds):,}')
print(f'Effective batch size: {effective_batch}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Model: Bahdanau Attention + Seq2Seq LSTM
# ─────────────────────────────────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention."""
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W_enc = nn.Linear(enc_hid, dec_hid, bias=False)
        self.W_dec = nn.Linear(dec_hid, dec_hid, bias=False)
        self.v     = nn.Linear(dec_hid, 1, bias=False)

    def forward(self, enc_out, dec_hidden, src_mask):
        # enc_out: (B, S, enc_hid) | dec_hidden: (B, dec_hid)
        energy = torch.tanh(
            self.W_enc(enc_out) + self.W_dec(dec_hidden).unsqueeze(1)
        )  # (B, S, dec_hid)
        scores = self.v(energy).squeeze(-1)  # (B, S)
        scores = scores.masked_fill(src_mask == 0, -1e4)
        attn_weights = F.softmax(scores, dim=-1)  # (B, S)
        context = (attn_weights.unsqueeze(2) * enc_out).sum(1)  # (B, enc_hid)
        return context, attn_weights


class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, bidirectional=True):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.rnn        = nn.LSTM(embed_dim, hidden_dim, n_layers,
                                  batch_first=True, dropout=dropout if n_layers > 1 else 0,
                                  bidirectional=bidirectional)
        self.dropout    = nn.Dropout(dropout)
        self.directions = 2 if bidirectional else 1
        self.n_layers   = n_layers
        self.hidden_dim = hidden_dim
        # Project bidirectional to hidden_dim for decoder init
        self.fc_h = nn.Linear(hidden_dim * self.directions, hidden_dim)
        self.fc_c = nn.Linear(hidden_dim * self.directions, hidden_dim)

    def forward(self, src, src_lens):
        embedded = self.dropout(self.embedding(src))  # (B, S, E)
        packed   = pack_padded_sequence(embedded, src_lens.cpu(), batch_first=True, enforce_sorted=True)
        out, (h, c) = self.rnn(packed)
        enc_out, _ = pad_packed_sequence(out, batch_first=True,total_length=src.size(1))  # (B, S, D*H)

        # h: (D*n_layers, B, H) — concat forward+backward for each layer
        h = h.view(self.n_layers, self.directions, -1, self.hidden_dim)
        c = c.view(self.n_layers, self.directions, -1, self.hidden_dim)
        # Take last layer, concat directions
        h_last = torch.cat([h[-1, 0], h[-1, 1]], dim=-1)  # (B, D*H)
        c_last = torch.cat([c[-1, 0], c[-1, 1]], dim=-1)
        h_init = torch.tanh(self.fc_h(h_last)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        c_init = torch.tanh(self.fc_c(c_last)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        return enc_out, (h_init, c_init)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hid, dec_hid, n_layers, dropout):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.attention  = BahdanauAttention(enc_hid, dec_hid)
        self.rnn        = nn.LSTM(embed_dim + enc_hid, dec_hid, n_layers,
                                  batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc_out     = nn.Linear(dec_hid + enc_hid + embed_dim, vocab_size)
        self.dropout    = nn.Dropout(dropout)

    def forward_step(self, token, hidden, enc_out, src_mask):
        # token: (B,) → (B, 1, E)
        embedded = self.dropout(self.embedding(token.unsqueeze(1)))
        dec_h = hidden[0][-1]  # last layer hidden: (B, H)
        context, attn_w = self.attention(enc_out, dec_h, src_mask)
        rnn_in = torch.cat([embedded, context.unsqueeze(1)], dim=-1)  # (B, 1, E+enc_hid)
        out, hidden = self.rnn(rnn_in, hidden)
        # Deep output combination (Luong-style)
        pred = self.fc_out(torch.cat([out.squeeze(1), context, embedded.squeeze(1)], dim=-1))
        return pred, hidden, attn_w

    def forward(self, tgt_in, enc_out, hidden, src_mask):
        B, T = tgt_in.shape
        logits = []
        token = tgt_in[:, 0]  # BOS
        for t in range(1, T):
            pred, hidden, _ = self.forward_step(token, hidden, enc_out, src_mask)
            logits.append(pred)
            token = tgt_in[:, t]  # teacher forcing
        return torch.stack(logits, dim=1)  # (B, T-1, V)


class Seq2Seq(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        enc_hid = cfg['hidden_dim'] * (2 if cfg['bidirectional'] else 1)
        self.encoder = Encoder(vocab_size, cfg['embed_dim'], cfg['hidden_dim'],
                               cfg['n_layers'], cfg['dropout'], cfg['bidirectional'])
        self.decoder = Decoder(vocab_size, cfg['embed_dim'], enc_hid,
                               cfg['hidden_dim'], cfg['n_layers'], cfg['dropout'])

    def forward(self, src, src_lens, tgt_in):
        enc_out, hidden = self.encoder(src, src_lens)
        src_mask = (src != PAD_ID)  # (B, S)
        logits = self.decoder(tgt_in, enc_out, hidden, src_mask)
        return logits  # (B, T-1, V)

    def beam_search(self, src, src_lens, beam_width=4, max_len=80, len_penalty=0.6):
        """Single-sentence beam search (batch_size=1)."""
        self.eval()
        with torch.no_grad():
            enc_out, hidden = self.encoder(src, src_lens)
            src_mask = (src != PAD_ID)

            beams = [([BOS_ID], hidden, 0.0)]  # (tokens, hidden, log_prob)
            completed = []

            for _ in range(max_len):
                new_beams = []
                for tokens, h, score in beams:
                    tok = torch.tensor([tokens[-1]], device=DEVICE)
                    logit, h_new, _ = self.decoder.forward_step(tok, h, enc_out, src_mask)
                    log_probs = F.log_softmax(logit, dim=-1).squeeze(0)
                    top_k = log_probs.topk(beam_width)
                    for lp, idx in zip(top_k.values, top_k.indices):
                        new_seq = tokens + [idx.item()]
                        new_score = score + lp.item()
                        if idx.item() == EOS_ID:
                            # Length-normalised score
                            norm = ((5 + len(new_seq)) / 6) ** len_penalty
                            completed.append((new_seq, new_score / norm))
                        else:
                            new_beams.append((new_seq, h_new, new_score))

                new_beams.sort(key=lambda x: x[2] / max(len(x[0]), 1), reverse=True)
                beams = new_beams[:beam_width]
                if not beams:
                    break

            if completed:
                completed.sort(key=lambda x: x[1], reverse=True)
                return completed[0][0][1:]  # strip BOS
            else:
                return beams[0][0][1:] if beams else [EOS_ID]


model = Seq2Seq(CFG, VOCAB_SIZE).to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Label-smoothed cross-entropy + optimiser + scheduler
# ─────────────────────────────────────────────────────────────────────────────
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, padding_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size  = vocab_size
        self.padding_idx = padding_idx
        self.smoothing   = smoothing
        self.confidence  = 1.0 - smoothing

    def forward(self, logits, targets):
        # logits: (B*T, V) | targets: (B*T,)
        B_T, V = logits.shape
        log_prob = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / (V - 2)  # exclude pad and true label
        with torch.no_grad():
            smooth_dist = torch.full_like(log_prob, smooth_val)
            smooth_dist[:, self.padding_idx] = 0
            smooth_dist.scatter_(1, targets.unsqueeze(1), self.confidence)
            mask = (targets == self.padding_idx)
            smooth_dist[mask] = 0
        loss = (-smooth_dist * log_prob).sum(dim=-1)
        n_non_pad = (~mask).sum().clamp(min=1)
        return loss.sum() / n_non_pad


criterion = LabelSmoothingLoss(VOCAB_SIZE, PAD_ID, CFG['label_smooth'])
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], betas=(0.9, 0.98), eps=1e-9)

def warmup_cosine(step, warmup=500, total=10000):
    if step < warmup:
        return step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return max(0.05, 0.5 * (1 + math.cos(math.pi * progress)))

total_steps = CFG['epochs'] * len(train_loader)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda s: warmup_cosine(s, CFG['warmup_steps'], total_steps)
)
scaler = GradScaler('cuda')

print('Loss fn, optimizer, scheduler ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — BLEU & CHRF++ evaluation
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(model_obj, loader, sp, device, n_samples=1000, beam_width=4):
    """Returns (bleu_100, chrf_100) using sacrebleu."""
    raw = model_obj if not isinstance(model_obj, nn.DataParallel) else model_obj.module
    raw.eval()

    hyps, refs = [], []
    count = 0
    with torch.no_grad():
        for src_pad, src_lens, tgt_in, tgt_out in loader:
            src_pad  = src_pad.to(device)
            src_lens = src_lens.to(device)
            B = src_pad.size(0)
            for i in range(B):
                s = src_pad[i:i+1]
                l = src_lens[i:i+1]
                pred_ids = raw.beam_search(s, l, beam_width=beam_width)
                # strip EOS and PAD
                pred_ids = [x for x in pred_ids if x not in (EOS_ID, PAD_ID, BOS_ID)]
                hyps.append(sp.decode(pred_ids))
                ref_ids = tgt_out[i].tolist()
                ref_ids = [x for x in ref_ids if x not in (EOS_ID, PAD_ID, BOS_ID)]
                refs.append(sp.decode(ref_ids))
                count += 1
                if count >= n_samples:
                    break
            if count >= n_samples:
                break

    bleu  = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf  = sacrebleu.corpus_chrf(hyps, [refs], beta=2).score
    return bleu, chrf, hyps[:5], refs[:5]


def eval_loss(model_obj, loader, criterion, device):
    model_obj.eval()
    total_loss, total_tok = 0, 0
    with torch.no_grad():
        for src_pad, src_lens, tgt_in, tgt_out in loader:
            src_pad  = src_pad.to(device)
            src_lens = src_lens.to(device)
            tgt_in   = tgt_in.to(device)
            tgt_out  = tgt_out.to(device)

            logits = model_obj(src_pad, src_lens, tgt_in)  # (B, T-1, V)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out[:, :T].reshape(B*T))
            n_tok = (tgt_out[:, :T] != PAD_ID).sum().item()
            total_loss += loss.item() * n_tok
            total_tok  += n_tok
    return total_loss / max(total_tok, 1)

print('Eval functions ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Training loop
# ─────────────────────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_bleu': [], 'val_bleu': [],
    'train_chrf': [], 'val_chrf': [],
}

best_val_bleu = -1
patience_counter = 0

def train_epoch(model_obj, loader, optimizer, criterion, scaler, scheduler, device, clip):
    model_obj.train()
    total_loss, total_tok = 0, 0
    for batch_idx, (src_pad, src_lens, tgt_in, tgt_out) in enumerate(
            tqdm(loader, desc='Training', leave=False)):
        src_pad  = src_pad.to(device, non_blocking=True)
        src_lens = src_lens.to(device, non_blocking=True)
        tgt_in   = tgt_in.to(device, non_blocking=True)
        tgt_out  = tgt_out.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda'):
            logits = model_obj(src_pad, src_lens, tgt_in)  # (B, T-1, V)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out[:, :T].reshape(B*T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_obj.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        n_tok = (tgt_out[:, :T] != PAD_ID).sum().item()
        total_loss += loss.item() * n_tok
        total_tok  += n_tok

    return total_loss / max(total_tok, 1)


print('Starting training...')
t0 = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    ep_t0 = time.time()

    # --- Train ---
    train_loss = train_epoch(model, train_loader, optimizer, criterion,
                             scaler, scheduler, DEVICE, CFG['clip_grad'])

    # --- Val loss ---
    val_loss = eval_loss(model, val_loader, criterion, DEVICE)

    # --- Metrics every eval_every epochs ---
    do_eval = (epoch % CFG['eval_every'] == 0)
    if do_eval:
        train_bleu, train_chrf, _, _ = compute_metrics(
            model, train_loader, sp, DEVICE, CFG['bleu_samples'], CFG['beam_width'])
        val_bleu, val_chrf, ex_hyps, ex_refs = compute_metrics(
            model, val_loader, sp, DEVICE, CFG['bleu_samples'], CFG['beam_width'])
    else:
        train_bleu = train_chrf = val_bleu = val_chrf = float('nan')

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_bleu'].append(train_bleu)
    history['val_bleu'].append(val_bleu)
    history['train_chrf'].append(train_chrf)
    history['val_chrf'].append(val_chrf)

    ep_time = time.time() - ep_t0
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{CFG["epochs"]} | '
          f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Val BLEU: {val_bleu:.2f} | Val CHRF++: {val_chrf:.2f} | '
          f'LR: {lr_now:.6f} | Time: {ep_time:.0f}s')

    # --- Checkpoint ---
    if do_eval and val_bleu > best_val_bleu:
        best_val_bleu = val_bleu
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state': (model.module if N_GPUS > 1 else model).state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_bleu': val_bleu,
            'val_chrf': val_chrf,
            'cfg': CFG,
            'history': history,
        }, CFG['ckpt_path'])
        print(f'  ✓ New best BLEU: {val_bleu:.2f} — checkpoint saved')
    elif do_eval:
        patience_counter += 1

    # --- Show sample translations ---
    if do_eval:
        print('  Sample translations:')
        for h, r in zip(ex_hyps[:3], ex_refs[:3]):
            print(f'    HYP: {h}')
            print(f'    REF: {r}')
            print()

print(f'\nTraining complete. Total time: {(time.time()-t0)/60:.1f} min')
print(f'Best Val BLEU: {best_val_bleu:.2f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Plots (loss, BLEU-100, CHRF++-100)
# ─────────────────────────────────────────────────────────────────────────────
epochs_x = list(range(1, CFG['epochs'] + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('NMT Training — Random Embeddings (LSTM + Bahdanau Attention)', fontsize=14, fontweight='bold')

# Loss
ax = axes[0]
ax.plot(epochs_x, history['train_loss'], 'b-o', ms=4, label='Train')
ax.plot(epochs_x, history['val_loss'],   'r-o', ms=4, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (label-smoothed CE)')
ax.set_title('Train & Val Loss'); ax.legend(); ax.grid(alpha=0.3)

# BLEU
ax = axes[1]
bleu_x = [e for e, v in zip(epochs_x, history['train_bleu']) if not math.isnan(v)]
ax.plot(bleu_x, [v for v in history['train_bleu'] if not math.isnan(v)], 'b-o', ms=4, label='Train')
ax.plot(bleu_x, [v for v in history['val_bleu']   if not math.isnan(v)], 'r-o', ms=4, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('BLEU-100')
ax.set_title('Train & Val BLEU-100'); ax.legend(); ax.grid(alpha=0.3)

# CHRF++
ax = axes[2]
ax.plot(bleu_x, [v for v in history['train_chrf'] if not math.isnan(v)], 'b-o', ms=4, label='Train')
ax.plot(bleu_x, [v for v in history['val_chrf']   if not math.isnan(v)], 'r-o', ms=4, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('CHRF++-100')
ax.set_title('Train & Val CHRF++-100'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CFG['out_dir'], 'random_emb_training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Final evaluation on best checkpoint + qualitative examples
# ─────────────────────────────────────────────────────────────────────────────
# Load best checkpoint
ckpt = torch.load(CFG['ckpt_path'], map_location=DEVICE)
raw_model = Seq2Seq(CFG, VOCAB_SIZE).to(DEVICE)
raw_model.load_state_dict(ckpt['model_state'])

val_bleu_final, val_chrf_final, hyps_final, refs_final = compute_metrics(
    raw_model, val_loader, sp, DEVICE, n_samples=2000, beam_width=CFG['beam_width'])

print('='*60)
print(f'FINAL RESULTS (Random Embeddings)')
print(f'  Val BLEU-100  : {val_bleu_final:.2f}')
print(f'  Val CHRF++-100: {val_chrf_final:.2f}')
print(f'  Best epoch    : {ckpt["epoch"]}')
print('='*60)
print('\nQualitative translations:')
for i, (h, r) in enumerate(zip(hyps_final[:10], refs_final[:10]), 1):
    print(f'  [{i}] HYP: {h}')
    print(f'      REF: {r}')
    print()

# Save results JSON
results = {
    'model': 'LSTM_RandomEmbeddings',
    'val_bleu_100': val_bleu_final,
    'val_chrf_100': val_chrf_final,
    'best_epoch': ckpt['epoch'],
    'history': history,
    'qualitative': [{'hyp': h, 'ref': r} for h, r in zip(hyps_final[:20], refs_final[:20])]
}
with open(os.path.join(CFG['out_dir'], 'random_emb_results.json'), 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('Results saved.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Hindi→Marathi & Marathi→Hindi inference demo
# ─────────────────────────────────────────────────────────────────────────────
def translate(model_obj, sentence, sp, device, beam_width=4, max_len=80):
    raw = model_obj if not isinstance(model_obj, nn.DataParallel) else model_obj.module
    raw.eval()
    ids = sp.encode(sentence)
    src = torch.tensor([ids], dtype=torch.long).to(device)
    src_lens = torch.tensor([len(ids)]).to(device)
    pred_ids = raw.beam_search(src, src_lens, beam_width=beam_width, max_len=max_len)
    pred_ids = [x for x in pred_ids if x not in (EOS_ID, PAD_ID, BOS_ID)]
    return sp.decode(pred_ids)


demo_sentences = [
    'मैं बाज़ार जा रहा हूँ।',
    'आज मौसम बहुत अच्छा है।',
    'वह स्कूल में पढ़ता है।',
    'हमें पानी पीना चाहिए।',
    'यह किताब बहुत रोचक है।'
]

print('Hindi → Marathi translations:')
print('─' * 60)
for sent in demo_sentences:
    trans = translate(raw_model, sent, sp, DEVICE)
    print(f'  HI: {sent}')
    print(f'  MR: {trans}')
    print()